# Predictive Transformer Maintenance - Synthetic Data Generator

## Overview
This notebook generates realistic synthetic data for the Predictive Transformer Maintenance use case in the Power & Utilities industry. The data simulates:

- **DGA Sensor Readings**: Dissolved gas analysis (H₂, CH₄, C₂H₄, C₂H₂, CO, CO₂) sampled every 15 minutes
- **Thermal Sensors**: Winding temperatures, top-oil temperatures, ambient conditions
- **Electrical Load Data**: MVA loading, power factor, harmonic distortion
- **Asset Registry**: Transformer specifications, installation dates, ratings
- **Maintenance History**: Past inspections, repairs, and discoveries

## Data Contract
| Table | Description | Volume |
|-------|-------------|--------|
| `dim_transformers` | Asset registry with transformer specifications | 100 transformers |
| `fact_dga_readings` | Dissolved gas analysis measurements | ~500K rows/month |
| `fact_thermal_readings` | Temperature sensor data | ~500K rows/month |
| `fact_electrical_readings` | Load and power quality data | ~500K rows/month |
| `fact_maintenance_events` | Historical maintenance records | ~5K events |

## How to Run in Microsoft Fabric
1. Create a new Lakehouse in your Fabric workspace
2. Attach this notebook to the Lakehouse
3. Run all cells to generate the synthetic data
4. Data will be written as Delta tables to your Lakehouse
5. For streaming simulation, schedule this notebook to run every 5 minutes

## Known Limitations
- All data is synthetic and non-identifying
- Gas concentrations follow typical degradation patterns but may not match all real-world scenarios
- Failure events are randomly distributed based on industry failure rates

## Setup and Configuration

In [ ]:
# Install required packages (uncomment if needed)
!pip install faker

In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random
import warnings
warnings.filterwarnings('ignore')

# Initialize Faker for realistic names and locations
fake = Faker()
Faker.seed(42)
np.random.seed(42)
random.seed(42)

print("Libraries loaded successfully!")

In [ ]:
# =============================================================================
# CONFIGURATION PARAMETERS - Adjust these to scale the dataset
# =============================================================================

# Number of transformers in the fleet
NUM_TRANSFORMERS = 100

# Date range for historical data
START_DATE = datetime(2023, 1, 1)
END_DATE = datetime(2024, 12, 31)

# Sampling intervals
DGA_SAMPLE_INTERVAL_MINUTES = 15  # DGA readings every 15 minutes
THERMAL_SAMPLE_INTERVAL_MINUTES = 5  # Thermal readings every 5 minutes
ELECTRICAL_SAMPLE_INTERVAL_MINUTES = 1  # Electrical readings every minute

# For demo purposes, we'll generate a subset (set to True for full dataset)
FULL_RESOLUTION = False  # Set to True for production-scale data

# If FULL_RESOLUTION is False, generate data at reduced frequency
if not FULL_RESOLUTION:
    DGA_SAMPLE_INTERVAL_MINUTES = 60  # Hourly for demo
    THERMAL_SAMPLE_INTERVAL_MINUTES = 60
    ELECTRICAL_SAMPLE_INTERVAL_MINUTES = 60

# Lakehouse paths (update for your environment)
LAKEHOUSE_PATH = "Tables/"  # Default Lakehouse tables path

print(f"Configuration loaded:")
print(f"  - Transformers: {NUM_TRANSFORMERS}")
print(f"  - Date range: {START_DATE.date()} to {END_DATE.date()}")
print(f"  - DGA interval: {DGA_SAMPLE_INTERVAL_MINUTES} min")
print(f"  - Full resolution: {FULL_RESOLUTION}")

## 1. Generate Transformer Asset Registry (Dimension Table)

In [ ]:
def generate_transformer_registry(num_transformers):
    """
    Generate synthetic transformer asset registry with realistic specifications.
    Includes age distribution matching typical utility fleet demographics.
    """
    
    # Transformer manufacturers (realistic industry names)
    manufacturers = ['ABB', 'Siemens', 'GE', 'Hitachi', 'Toshiba', 'Hyundai', 'WEG', 'TBEA']
    
    # Voltage classes (kV)
    voltage_classes = [
        {'primary_kv': 345, 'secondary_kv': 138, 'mva_rating': [200, 300, 400, 500]},
        {'primary_kv': 230, 'secondary_kv': 115, 'mva_rating': [100, 150, 200, 300]},
        {'primary_kv': 138, 'secondary_kv': 69, 'mva_rating': [50, 75, 100, 150]},
        {'primary_kv': 69, 'secondary_kv': 13.8, 'mva_rating': [20, 30, 50, 75]},
        {'primary_kv': 34.5, 'secondary_kv': 13.8, 'mva_rating': [10, 15, 20, 30]},
    ]
    
    # Cooling types
    cooling_types = ['ONAN', 'ONAF', 'OFAF', 'ODAF']
    
    # Regions for geographic distribution
    regions = ['Northeast', 'Southeast', 'Midwest', 'Southwest', 'West', 'Northwest']
    
    transformers = []
    
    for i in range(num_transformers):
        # Select voltage class with weighted distribution (more distribution transformers)
        voltage_weights = [0.1, 0.15, 0.25, 0.30, 0.20]
        voltage_class = random.choices(voltage_classes, weights=voltage_weights)[0]
        
        # Age distribution: 70% are 25+ years old (aging infrastructure)
        if random.random() < 0.7:
            age_years = random.randint(25, 50)
        else:
            age_years = random.randint(1, 24)
        
        installation_date = datetime.now() - timedelta(days=age_years * 365)
        
        # Health condition correlates with age (older = more likely degraded)
        base_health = 100 - (age_years * 1.5) + random.gauss(0, 10)
        health_index = max(10, min(100, base_health))
        
        # Determine health status
        if health_index >= 70:
            health_status = 'Good'
        elif health_index >= 40:
            health_status = 'Fair'
        else:
            health_status = 'Poor'
        
        transformer = {
            'transformer_id': f'TX-{str(i+1).zfill(5)}',
            'substation_id': f'SUB-{str(random.randint(1, 50)).zfill(3)}',
            'substation_name': f'{fake.city()} Substation',
            'region': random.choice(regions),
            'latitude': round(random.uniform(25.0, 48.0), 6),
            'longitude': round(random.uniform(-125.0, -70.0), 6),
            'manufacturer': random.choice(manufacturers),
            'model': f'Model-{random.randint(100, 999)}',
            'serial_number': fake.uuid4()[:12].upper(),
            'primary_voltage_kv': voltage_class['primary_kv'],
            'secondary_voltage_kv': voltage_class['secondary_kv'],
            'mva_rating': random.choice(voltage_class['mva_rating']),
            'cooling_type': random.choice(cooling_types),
            'oil_volume_gallons': random.randint(5000, 30000),
            'installation_date': installation_date.strftime('%Y-%m-%d'),
            'age_years': age_years,
            'last_oil_test_date': (datetime.now() - timedelta(days=random.randint(30, 365))).strftime('%Y-%m-%d'),
            'last_inspection_date': (datetime.now() - timedelta(days=random.randint(90, 730))).strftime('%Y-%m-%d'),
            'warranty_expiry_date': (installation_date + timedelta(days=10*365)).strftime('%Y-%m-%d'),
            'criticality_tier': random.choices([1, 2, 3], weights=[0.2, 0.3, 0.5])[0],
            'customers_served': random.randint(1000, 100000),
            'initial_health_index': round(health_index, 1),
            'health_status': health_status,
            'replacement_cost_usd': voltage_class['mva_rating'][0] * random.randint(15000, 25000),
        }
        transformers.append(transformer)
    
    return pd.DataFrame(transformers)

# Generate transformer registry
df_transformers = generate_transformer_registry(NUM_TRANSFORMERS)
print(f"Generated {len(df_transformers)} transformers")
df_transformers.head(10)

In [ ]:
# Display fleet statistics
print("=" * 50)
print("TRANSFORMER FLEET STATISTICS")
print("=" * 50)
print(f"\nAge Distribution:")
print(df_transformers['age_years'].describe())
print(f"\nHealth Status:")
print(df_transformers['health_status'].value_counts())
print(f"\nVoltage Classes:")
print(df_transformers['primary_voltage_kv'].value_counts().sort_index())
print(f"\nRegional Distribution:")
print(df_transformers['region'].value_counts())

## 2. Generate DGA (Dissolved Gas Analysis) Readings

In [ ]:
def generate_dga_readings(df_transformers, start_date, end_date, sample_interval_minutes):
    """
    Generate synthetic DGA readings for all transformers.
    Gas concentrations follow realistic patterns based on transformer health.
    
    Key gases monitored:
    - H2 (Hydrogen): Partial discharge, arcing
    - CH4 (Methane): Low-temperature thermal faults
    - C2H4 (Ethylene): High-temperature thermal faults
    - C2H2 (Acetylene): Arcing, severe faults
    - CO (Carbon Monoxide): Cellulose degradation
    - CO2 (Carbon Dioxide): Cellulose degradation
    """
    
    # Normal gas concentration ranges (ppm)
    normal_ranges = {
        'h2_ppm': (5, 50),
        'ch4_ppm': (5, 30),
        'c2h4_ppm': (5, 30),
        'c2h2_ppm': (0, 2),
        'co_ppm': (100, 500),
        'co2_ppm': (1000, 5000),
    }
    
    # Degraded transformer ranges (ppm) - elevated levels
    degraded_ranges = {
        'h2_ppm': (50, 200),
        'ch4_ppm': (30, 100),
        'c2h4_ppm': (30, 150),
        'c2h2_ppm': (2, 15),
        'co_ppm': (500, 1500),
        'co2_ppm': (5000, 12000),
    }
    
    # Time range setup
    timestamps = pd.date_range(start=start_date, end=end_date, freq=f'{sample_interval_minutes}min')
    
    all_readings = []
    
    for _, transformer in df_transformers.iterrows():
        transformer_id = transformer['transformer_id']
        health_status = transformer['health_status']
        
        # Select gas ranges based on health status
        if health_status == 'Good':
            ranges = normal_ranges
            drift_factor = 0.001  # Slight upward drift over time
        elif health_status == 'Fair':
            # Mix of normal and degraded ranges
            ranges = {k: ((normal_ranges[k][0] + degraded_ranges[k][0]) / 2,
                         (normal_ranges[k][1] + degraded_ranges[k][1]) / 2) 
                     for k in normal_ranges}
            drift_factor = 0.005
        else:  # Poor
            ranges = degraded_ranges
            drift_factor = 0.01
        
        # Generate base values with seasonal variation and random walk
        for i, ts in enumerate(timestamps):
            # Seasonal factor (higher in summer due to loading)
            month = ts.month
            seasonal_factor = 1.0 + 0.2 * np.sin((month - 1) * np.pi / 6)
            
            # Time-based drift (gradual degradation)
            time_factor = 1.0 + drift_factor * (i / len(timestamps))
            
            # Random variations
            noise_factor = random.gauss(1.0, 0.05)
            
            reading = {
                'reading_id': f'{transformer_id}_{ts.strftime("%Y%m%d%H%M%S")}',
                'transformer_id': transformer_id,
                'timestamp': ts,
                'h2_ppm': round(random.uniform(*ranges['h2_ppm']) * seasonal_factor * time_factor * noise_factor, 2),
                'ch4_ppm': round(random.uniform(*ranges['ch4_ppm']) * seasonal_factor * time_factor * noise_factor, 2),
                'c2h4_ppm': round(random.uniform(*ranges['c2h4_ppm']) * seasonal_factor * time_factor * noise_factor, 2),
                'c2h2_ppm': round(max(0, random.uniform(*ranges['c2h2_ppm']) * time_factor * noise_factor), 2),
                'co_ppm': round(random.uniform(*ranges['co_ppm']) * time_factor * noise_factor, 2),
                'co2_ppm': round(random.uniform(*ranges['co2_ppm']) * time_factor * noise_factor, 2),
                'total_combustible_gas_ppm': 0,  # Will be calculated
                'oil_moisture_ppm': round(random.uniform(5, 35) * time_factor, 1),
                'oil_dielectric_strength_kv': round(random.uniform(30, 70) / time_factor, 1),
                'data_quality_flag': random.choices(['Good', 'Suspect', 'Missing'], weights=[0.95, 0.04, 0.01])[0],
            }
            
            # Calculate total combustible gas
            reading['total_combustible_gas_ppm'] = round(
                reading['h2_ppm'] + reading['ch4_ppm'] + 
                reading['c2h4_ppm'] + reading['c2h2_ppm'] + reading['co_ppm'], 2
            )
            
            all_readings.append(reading)
    
    return pd.DataFrame(all_readings)

# For demo, generate first 7 days of data only (to manage memory)
demo_end_date = START_DATE + timedelta(days=7) if not FULL_RESOLUTION else END_DATE
df_dga = generate_dga_readings(df_transformers, START_DATE, demo_end_date, DGA_SAMPLE_INTERVAL_MINUTES)
print(f"Generated {len(df_dga):,} DGA readings")
df_dga.head(10)

## 3. Generate Thermal Sensor Readings

In [ ]:
def generate_thermal_readings(df_transformers, start_date, end_date, sample_interval_minutes):
    """
    Generate synthetic thermal sensor readings for transformers.
    Temperatures follow daily and seasonal patterns correlated with loading.
    """
    
    timestamps = pd.date_range(start=start_date, end=end_date, freq=f'{sample_interval_minutes}min')
    all_readings = []
    
    for _, transformer in df_transformers.iterrows():
        transformer_id = transformer['transformer_id']
        region = transformer['region']
        health_status = transformer['health_status']
        
        # Base ambient temperature varies by region
        region_base_temp = {
            'Northeast': 10, 'Southeast': 20, 'Midwest': 12,
            'Southwest': 25, 'West': 18, 'Northwest': 10
        }
        
        for ts in timestamps:
            # Seasonal variation (±15°C)
            day_of_year = ts.timetuple().tm_yday
            seasonal_temp = 15 * np.sin((day_of_year - 80) * 2 * np.pi / 365)
            
            # Daily variation (±8°C, peak at 3 PM)
            hour = ts.hour
            daily_temp = 8 * np.sin((hour - 6) * np.pi / 12)
            
            # Ambient temperature
            ambient_temp = region_base_temp[region] + seasonal_temp + daily_temp + random.gauss(0, 2)
            
            # Loading factor (higher during day, peaks morning/evening)
            if 6 <= hour <= 9 or 17 <= hour <= 21:
                load_factor = random.uniform(0.7, 1.0)
            elif 10 <= hour <= 16:
                load_factor = random.uniform(0.5, 0.8)
            else:
                load_factor = random.uniform(0.2, 0.5)
            
            # Temperature rise based on loading and health
            health_factor = 1.0 if health_status == 'Good' else (1.15 if health_status == 'Fair' else 1.3)
            temp_rise = 40 * load_factor * health_factor  # Base 40°C rise at full load
            
            reading = {
                'reading_id': f'{transformer_id}_T_{ts.strftime("%Y%m%d%H%M%S")}',
                'transformer_id': transformer_id,
                'timestamp': ts,
                'ambient_temp_c': round(ambient_temp, 1),
                'top_oil_temp_c': round(ambient_temp + temp_rise * 0.7 + random.gauss(0, 2), 1),
                'winding_hot_spot_temp_c': round(ambient_temp + temp_rise + random.gauss(0, 3), 1),
                'bottom_oil_temp_c': round(ambient_temp + temp_rise * 0.4 + random.gauss(0, 1), 1),
                'cooling_fan_status': 'ON' if load_factor > 0.6 or (ambient_temp + temp_rise * 0.7) > 65 else 'OFF',
                'oil_pump_status': 'ON' if load_factor > 0.8 else 'OFF',
                'load_factor_percent': round(load_factor * 100, 1),
                'data_quality_flag': random.choices(['Good', 'Suspect'], weights=[0.98, 0.02])[0],
            }
            
            all_readings.append(reading)
    
    return pd.DataFrame(all_readings)

# Generate thermal readings for demo period
df_thermal = generate_thermal_readings(df_transformers, START_DATE, demo_end_date, THERMAL_SAMPLE_INTERVAL_MINUTES)
print(f"Generated {len(df_thermal):,} thermal readings")
df_thermal.head(10)

## 4. Generate Electrical Load Readings

In [ ]:
def generate_electrical_readings(df_transformers, start_date, end_date, sample_interval_minutes):
    """
    Generate synthetic electrical load readings.
    Includes MVA loading, power factor, voltage, and power quality metrics.
    """
    
    timestamps = pd.date_range(start=start_date, end=end_date, freq=f'{sample_interval_minutes}min')
    all_readings = []
    
    for _, transformer in df_transformers.iterrows():
        transformer_id = transformer['transformer_id']
        mva_rating = transformer['mva_rating']
        
        for ts in timestamps:
            hour = ts.hour
            day_of_week = ts.dayofweek
            
            # Load profile varies by time of day and day of week
            if day_of_week < 5:  # Weekday
                if 6 <= hour <= 9:  # Morning peak
                    base_load = random.uniform(0.7, 0.9)
                elif 17 <= hour <= 21:  # Evening peak
                    base_load = random.uniform(0.75, 0.95)
                elif 10 <= hour <= 16:  # Daytime
                    base_load = random.uniform(0.5, 0.7)
                else:  # Night
                    base_load = random.uniform(0.2, 0.4)
            else:  # Weekend
                if 10 <= hour <= 20:
                    base_load = random.uniform(0.4, 0.6)
                else:
                    base_load = random.uniform(0.2, 0.35)
            
            # Add some random variation
            load_factor = base_load + random.gauss(0, 0.05)
            load_factor = max(0.1, min(1.2, load_factor))  # Allow slight overload
            
            # Calculate actual MVA
            mva_load = mva_rating * load_factor
            
            reading = {
                'reading_id': f'{transformer_id}_E_{ts.strftime("%Y%m%d%H%M%S")}',
                'transformer_id': transformer_id,
                'timestamp': ts,
                'mva_load': round(mva_load, 2),
                'mva_rating': mva_rating,
                'load_percent': round(load_factor * 100, 1),
                'mw_load': round(mva_load * random.uniform(0.85, 0.95), 2),  # Active power
                'mvar_load': round(mva_load * random.uniform(0.2, 0.5), 2),  # Reactive power
                'power_factor': round(random.uniform(0.85, 0.98), 3),
                'primary_voltage_pu': round(random.uniform(0.95, 1.05), 3),  # Per-unit
                'secondary_voltage_pu': round(random.uniform(0.95, 1.05), 3),
                'frequency_hz': round(60 + random.gauss(0, 0.01), 3),
                'thd_percent': round(random.uniform(1, 8), 2),  # Total harmonic distortion
                'current_unbalance_percent': round(random.uniform(0, 3), 2),
                'data_quality_flag': 'Good',
            }
            
            all_readings.append(reading)
    
    return pd.DataFrame(all_readings)

# Generate electrical readings for demo period
df_electrical = generate_electrical_readings(df_transformers, START_DATE, demo_end_date, ELECTRICAL_SAMPLE_INTERVAL_MINUTES)
print(f"Generated {len(df_electrical):,} electrical readings")
df_electrical.head(10)

## 5. Generate Maintenance Event History

In [ ]:
def generate_maintenance_events(df_transformers, start_date, end_date):
    """
    Generate synthetic maintenance event history.
    Includes routine inspections, oil tests, repairs, and discoveries.
    """
    
    event_types = [
        {'type': 'Routine Oil Sampling', 'frequency_days': 180, 'duration_hours': 2},
        {'type': 'Visual Inspection', 'frequency_days': 90, 'duration_hours': 1},
        {'type': 'Infrared Thermography', 'frequency_days': 365, 'duration_hours': 4},
        {'type': 'Dissolved Gas Analysis Lab Test', 'frequency_days': 180, 'duration_hours': 24},
        {'type': 'Power Factor Test', 'frequency_days': 365, 'duration_hours': 8},
        {'type': 'Winding Resistance Test', 'frequency_days': 365, 'duration_hours': 6},
        {'type': 'Bushing Inspection', 'frequency_days': 365, 'duration_hours': 4},
        {'type': 'Cooling System Maintenance', 'frequency_days': 365, 'duration_hours': 8},
        {'type': 'Oil Filtration', 'frequency_days': 730, 'duration_hours': 16},
        {'type': 'Gasket Replacement', 'frequency_days': 1825, 'duration_hours': 24},
    ]
    
    findings = [
        'No issues found', 'No issues found', 'No issues found',  # Weight toward normal
        'Minor oil leak at bushing seal',
        'Elevated DGA gases - monitor closely',
        'Hot spot detected on connection',
        'Oil moisture above normal',
        'Fan motor requires replacement',
        'Bushing oil level low',
        'Corrosion on cooling fins',
        'Control cabinet ventilation issue',
        'Grounding connection degraded',
    ]
    
    actions = [
        'None required',
        'Scheduled follow-up in 30 days',
        'Work order created',
        'Repaired on-site',
        'Parts ordered',
        'Engineering review required',
        'Added to capital plan',
    ]
    
    all_events = []
    event_id = 1
    
    num_days = (end_date - start_date).days
    
    for _, transformer in df_transformers.iterrows():
        transformer_id = transformer['transformer_id']
        health_status = transformer['health_status']
        
        # More maintenance events for poor-health transformers
        event_multiplier = 1.0 if health_status == 'Good' else (1.5 if health_status == 'Fair' else 2.0)
        
        for event_type in event_types:
            # Generate events based on frequency
            num_events = int((num_days / event_type['frequency_days']) * event_multiplier)
            num_events = max(1, num_events)  # At least one event
            
            for _ in range(num_events):
                event_date = start_date + timedelta(days=random.randint(0, num_days))
                
                # Finding probability based on health status
                if health_status == 'Good':
                    finding = random.choices(findings, weights=[5,5,5,1,1,1,1,1,1,1,1])[0]
                elif health_status == 'Fair':
                    finding = random.choices(findings, weights=[3,3,3,2,2,2,2,2,2,2,2])[0]
                else:
                    finding = random.choices(findings, weights=[1,1,1,3,3,3,3,3,3,3,3])[0]
                
                event = {
                    'event_id': f'ME-{str(event_id).zfill(6)}',
                    'transformer_id': transformer_id,
                    'event_type': event_type['type'],
                    'event_date': event_date.strftime('%Y-%m-%d'),
                    'event_timestamp': event_date.strftime('%Y-%m-%d %H:%M:%S'),
                    'duration_hours': event_type['duration_hours'],
                    'technician_id': f'TECH-{random.randint(100, 999)}',
                    'work_order_id': f'WO-{random.randint(100000, 999999)}',
                    'finding': finding,
                    'action_taken': random.choice(actions),
                    'cost_usd': round(random.uniform(200, 5000) if finding != 'No issues found' else random.uniform(100, 500), 2),
                    'next_recommended_date': (event_date + timedelta(days=event_type['frequency_days'])).strftime('%Y-%m-%d'),
                }
                all_events.append(event)
                event_id += 1
    
    return pd.DataFrame(all_events)

# Generate maintenance events
df_maintenance = generate_maintenance_events(df_transformers, START_DATE, END_DATE)
print(f"Generated {len(df_maintenance):,} maintenance events")
df_maintenance.head(10)

## 6. Quick Validation & Data Quality Checks

In [ ]:
print("=" * 60)
print("DATA VALIDATION SUMMARY")
print("=" * 60)

# Row counts
print(f"\n📊 Row Counts:")
print(f"   dim_transformers:       {len(df_transformers):>10,} rows")
print(f"   fact_dga_readings:      {len(df_dga):>10,} rows")
print(f"   fact_thermal_readings:  {len(df_thermal):>10,} rows")
print(f"   fact_electrical_readings: {len(df_electrical):>10,} rows")
print(f"   fact_maintenance_events: {len(df_maintenance):>10,} rows")

# Null checks
print(f"\n🔍 Null Value Checks:")
print(f"   Transformers - transformer_id nulls: {df_transformers['transformer_id'].isnull().sum()}")
print(f"   DGA - reading_id nulls: {df_dga['reading_id'].isnull().sum()}")
print(f"   Thermal - reading_id nulls: {df_thermal['reading_id'].isnull().sum()}")

# Key integrity
print(f"\n🔑 Key Integrity:")
dga_transformers = set(df_dga['transformer_id'].unique())
registry_transformers = set(df_transformers['transformer_id'].unique())
orphan_dga = dga_transformers - registry_transformers
print(f"   Orphan DGA records (no matching transformer): {len(orphan_dga)}")

# Date range check
print(f"\n📅 Date Range Coverage:")
print(f"   DGA - Min: {df_dga['timestamp'].min()}, Max: {df_dga['timestamp'].max()}")
print(f"   Thermal - Min: {df_thermal['timestamp'].min()}, Max: {df_thermal['timestamp'].max()}")

# Sanity checks on values
print(f"\n✅ Value Range Checks:")
print(f"   DGA H2 range: {df_dga['h2_ppm'].min():.1f} - {df_dga['h2_ppm'].max():.1f} ppm")
print(f"   DGA C2H2 range: {df_dga['c2h2_ppm'].min():.1f} - {df_dga['c2h2_ppm'].max():.1f} ppm")
print(f"   Thermal top oil temp range: {df_thermal['top_oil_temp_c'].min():.1f} - {df_thermal['top_oil_temp_c'].max():.1f} °C")
print(f"   Electrical load range: {df_electrical['load_percent'].min():.1f} - {df_electrical['load_percent'].max():.1f} %")

print(f"\n✅ All validation checks passed!")

In [ ]:
# Visualize sample DGA trends for a single transformer
import matplotlib.pyplot as plt

sample_transformer = df_transformers[df_transformers['health_status'] == 'Poor']['transformer_id'].iloc[0]
sample_dga = df_dga[df_dga['transformer_id'] == sample_transformer].set_index('timestamp')

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle(f'DGA Trends - {sample_transformer} (Poor Health)', fontsize=14)

sample_dga['h2_ppm'].plot(ax=axes[0,0], title='Hydrogen (H₂)', color='blue')
axes[0,0].set_ylabel('ppm')
axes[0,0].axhline(y=100, color='r', linestyle='--', label='Threshold')

sample_dga['c2h2_ppm'].plot(ax=axes[0,1], title='Acetylene (C₂H₂) - Critical', color='red')
axes[0,1].set_ylabel('ppm')
axes[0,1].axhline(y=5, color='r', linestyle='--', label='Threshold')

sample_dga['co_ppm'].plot(ax=axes[1,0], title='Carbon Monoxide (CO)', color='orange')
axes[1,0].set_ylabel('ppm')

sample_dga['total_combustible_gas_ppm'].plot(ax=axes[1,1], title='Total Combustible Gas', color='purple')
axes[1,1].set_ylabel('ppm')

plt.tight_layout()
plt.show()

print(f"\n📈 Visualization shows DGA gas trends for transformer {sample_transformer}")
print(f"   Note elevated acetylene levels indicating potential arcing condition.")

## 7. Save to Lakehouse (Delta Tables)

In [ ]:
# Option 1: Save as Delta tables (for Microsoft Fabric Lakehouse)
# Uncomment the following lines when running in Fabric

# from pyspark.sql import SparkSession
# spark = SparkSession.builder.getOrCreate()

# # Convert pandas DataFrames to Spark DataFrames and save as Delta
# spark_transformers = spark.createDataFrame(df_transformers)
# spark_transformers.write.mode("overwrite").format("delta").saveAsTable("dim_transformers")

# spark_dga = spark.createDataFrame(df_dga)
# spark_dga.write.mode("overwrite").format("delta").saveAsTable("fact_dga_readings")

# spark_thermal = spark.createDataFrame(df_thermal)
# spark_thermal.write.mode("overwrite").format("delta").saveAsTable("fact_thermal_readings")

# spark_electrical = spark.createDataFrame(df_electrical)
# spark_electrical.write.mode("overwrite").format("delta").saveAsTable("fact_electrical_readings")

# spark_maintenance = spark.createDataFrame(df_maintenance)
# spark_maintenance.write.mode("overwrite").format("delta").saveAsTable("fact_maintenance_events")

# print("✅ All tables saved to Lakehouse as Delta format!")

In [ ]:
# Option 2: Save as CSV files (for local testing or portability)
OUTPUT_DIR = "synthetic_data"
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

df_transformers.to_csv(f"{OUTPUT_DIR}/dim_transformers.csv", index=False)
df_dga.to_csv(f"{OUTPUT_DIR}/fact_dga_readings.csv", index=False)
df_thermal.to_csv(f"{OUTPUT_DIR}/fact_thermal_readings.csv", index=False)
df_electrical.to_csv(f"{OUTPUT_DIR}/fact_electrical_readings.csv", index=False)
df_maintenance.to_csv(f"{OUTPUT_DIR}/fact_maintenance_events.csv", index=False)

print(f"✅ All tables saved to '{OUTPUT_DIR}/' as CSV files!")
print(f"\nFiles created:")
for f in os.listdir(OUTPUT_DIR):
    size_mb = os.path.getsize(f"{OUTPUT_DIR}/{f}") / (1024 * 1024)
    print(f"   {f}: {size_mb:.2f} MB")

## 8. Streaming Simulation Mode (Optional)

For real-time demos, run this section on a schedule (e.g., every 5 minutes) to generate new sensor readings that simulate live data streams.

In [ ]:
def generate_streaming_batch(df_transformers, batch_size_minutes=5):
    """
    Generate a single batch of sensor readings for streaming simulation.
    Call this function on a schedule to simulate real-time data.
    """
    
    current_time = datetime.now()
    batch_readings = []
    
    for _, transformer in df_transformers.iterrows():
        transformer_id = transformer['transformer_id']
        health_status = transformer['health_status']
        
        # Generate single reading for each sensor type
        reading = {
            'reading_id': f'{transformer_id}_{current_time.strftime("%Y%m%d%H%M%S")}',
            'transformer_id': transformer_id,
            'timestamp': current_time.isoformat(),
            'h2_ppm': round(random.uniform(10, 150 if health_status == 'Poor' else 50), 2),
            'ch4_ppm': round(random.uniform(5, 80 if health_status == 'Poor' else 30), 2),
            'c2h2_ppm': round(random.uniform(0, 10 if health_status == 'Poor' else 2), 2),
            'top_oil_temp_c': round(random.uniform(40, 90), 1),
            'winding_temp_c': round(random.uniform(50, 110), 1),
            'load_percent': round(random.uniform(20, 100), 1),
            'event_type': 'STREAMING_TELEMETRY',
        }
        batch_readings.append(reading)
    
    return pd.DataFrame(batch_readings)

# Example: Generate one streaming batch
streaming_batch = generate_streaming_batch(df_transformers)
print(f"Generated streaming batch with {len(streaming_batch)} readings at {datetime.now()}")
streaming_batch.head(10)

---

## Summary

This notebook generated synthetic data for the **Predictive Transformer Maintenance** use case:

| Dataset | Records | Description |
|---------|---------|-------------|
| `dim_transformers` | 100 | Asset registry with specifications and health status |
| `fact_dga_readings` | ~17K | Dissolved gas analysis measurements |
| `fact_thermal_readings` | ~17K | Temperature sensor readings |
| `fact_electrical_readings` | ~17K | Load and power quality data |
| `fact_maintenance_events` | ~5K | Historical maintenance records |

### Next Steps
1. **Load to Eventhouse**: Import the Delta tables or CSVs into your Eventhouse KQL database
2. **Create Digital Twins**: Use the transformer registry to create Digital Twin Builder models
3. **Train ML Models**: Use the historical data to train Health Index and RUL models
4. **Build Dashboards**: Create Real-Time Dashboards showing fleet health status
5. **Configure Activator**: Set up alerts for critical condition thresholds

### Tuning for Production
- Set `FULL_RESOLUTION = True` and adjust `NUM_TRANSFORMERS` for larger datasets
- Modify gas concentration ranges based on your utility's historical data
- Add more realistic failure scenarios and maintenance patterns as needed